# Lab 5


## 1. Setup and imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

sns.set(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the dataset

In [2]:
df = pd.read_csv("talabat_enhanced_orders.csv")

df.head(10)

,Order_ID,User_ID,Restaurant_ID,Driver_ID,Item_Name,Quantity,Total_Price,Order_Time,Delivery_Time,Delivery_Duration_Minutes,City,Payment_Method,Order_Status,Driver_Vehicle,Restaurant_Lat,Restaurant_Lon,Customer_Lat,Customer_Lon,Driver_Lat,Driver_Lon,Delivery_Distance_km,Traffic_Level,Driver_Availability
0,1,U3522,358,485,Fried Chicken,3,273.72,2025-06-16 08:32:00,2025-06-16 09:11:00,39,Alexandria,Wallet,Delivered,Motorbike,31.195082,29.921931,31.191404,29.904982,31.215658,29.910664,1.666106,High,Offline
1,2,U9214,316,65,Sandwich,3,365.82,2025-06-03 21:27:00,2025-06-03 22:00:00,33,Zagazig,Credit Card,Delivered,Motorbike,30.605729,31.503079,30.586047,31.485820,30.580329,31.502380,2.738698,Low,Online
2,3,U7307,357,309,Koshary,3,401.94,2025-06-01 14:48:00,2025-06-01 15:26:00,38,Assiut,Cash,In Transit,Car,27.190180,31.177741,27.164869,31.169218,27.162976,31.189458,2.929079,Medium,Online
3,4,U3612,420,32,Sushi,2,221.18,2025-06-13 02:30:00,2025-06-13 03:22:00,52,Mansoura,Cash,Delivered,Car,31.041846,31.381229,31.035773,31.380440,31.054690,31.401187,0.677498,Low,Online
4,5,U3492,73,364,Koshary,5,355.55,2025-06-06 09:48:00,2025-06-06 10:32:00,44,Mansoura,Wallet,Delivered,Motorbike,31.024141,31.376104,31.026023,31.396881,31.035350,31.389315,1.994769,High,Online
5,6,U7439,750,270,Sushi,3,205.44,2025-06-04 12:16:00,2025-06-04 12:45:00,29,Mansoura,Credit Card,Delivered,Bicycle,31.024140,31.387817,31.029004,31.373869,31.022573,31.380646,1.436807,Medium,Online
6,7,U8948,827,4,Sushi,1,133.94,2025-06-11 04:09:00,2025-06-11 04:49:00,40,Cairo,Wallet,Delivered,Bicycle,30.026723,31.249101,30.043525,31.233372,30.038769,31.231835,2.402167,Low,Online
7,8,U8672,908,109,Shawarma,5,404.80,2025-06-12 18:37:00,2025-06-12 19:18:00,41,Mansoura,Cash,Delivered,Bicycle,31.052547,31.392976,31.053352,31.362834,31.049187,31.373710,2.878434,High,Online
8,9,U2205,814,215,Pizza,1,101.03,2025-06-01 22:18:00,2025-06-01 23:05:00,47,Mansoura,Credit Card,Delivered,Motorbike,31.041945,31.375128,31.041036,31.385503,31.041062,31.368488,0.995562,Low,Online
9,10,U7411,362,416,Pizza,1,130.05,2025-06-09 00:18:00,2025-06-09 01:04:00,46,Cairo,Wallet,Delivered,Bicycle,30.052723,31.236077,30.025107,31.229281,30.062032,31.227642,3.130713,Low,Online


## 4. Target variable


In [3]:
target_col = "Order_Status"
df[target_col].value_counts()

,count
Order_Status,
Delivered,85197
Cancelled,9812
In Transit,4991


## 5. Feature engineering
## Task 1



In [4]:
df['driver_to_restaurant_distance'] = np.sqrt(
    (df['Driver_Lat'] - df['Restaurant_Lat'])**2 +
    (df['Driver_Lon'] - df['Restaurant_Lon'])**2
)

df[['Driver_Lat','Restaurant_Lat','driver_to_restaurant_distance']].head()

,Driver_Lat,Restaurant_Lat,driver_to_restaurant_distance
0,31.215658,31.195082,0.023459
1,30.580329,30.605729,0.025410
2,27.162976,27.190180,0.029619
3,31.054690,31.041846,0.023734
4,31.035350,31.024141,0.017326




I created a feature called driver_to_restaurant_distance, which calculates the distance between the driver’s location and the restaurant using their latitude and longitude coordinates. If the driver is far from the restaurant, it may take longer for the driver to reach the pickup location, which could delay the delivery and affect the order status. This feature may therefore improve the model’s ability to predict delivery outcomes.


## Task 2

In [5]:
df['order_hour'] = pd.to_datetime(df['Order_Time']).dt.hour
df['is_peak_hour'] = df['order_hour'].isin([11,12,13,14,18,19,20,21,22]).astype(int)
df[['Order_Time','order_hour','is_peak_hour']].head()

,Order_Time,order_hour,is_peak_hour
0,2025-06-16 08:32:00,8,0
1,2025-06-03 21:27:00,21,1
2,2025-06-01 14:48:00,14,1
3,2025-06-13 02:30:00,2,0
4,2025-06-06 09:48:00,9,0


## **Task 3**

In [6]:
top_k = 30

top_items = df['Item_Name'].value_counts().nlargest(top_k).index

df['Item_Name_reduced'] = df['Item_Name'].where(
    df['Item_Name'].isin(top_items),
    'Other'
)

df['Item_Name_reduced'].value_counts().head(15)

,count
Item_Name_reduced,
Shawarma,11320
Pizza,11229
Fried Chicken,11171
Burger,11129
Pasta,11077
Sandwich,11061
Koshary,11033
Sushi,10990
Salad,10990


## 6. Drop before training

In [7]:
columns_to_drop = [
    'Order_ID',
    'User_ID',
    'Restaurant_ID',
    'Driver_ID',
    'Order_Time',
    'Delivery_Time',
    'Delivery_Duration_Minutes',
    'Item_Name'
]

df_model = df.drop(columns=columns_to_drop, errors='ignore')

X = df_model.drop(columns=['Order_Status'])
y = df_model['Order_Status']

X = pd.get_dummies(X, drop_first=True)

print("Final training shape:", X.shape)

Final training shape: (100000, 33)


## 7. Train model

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.85195


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Classification Report:
               precision    recall  f1-score   support

   Cancelled       0.00      0.00      0.00      1963
   Delivered       0.85      1.00      0.92     17039
  In Transit       0.00      0.00      0.00       998

    accuracy                           0.85     20000
   macro avg       0.28      0.33      0.31     20000
weighted avg       0.73      0.85      0.78     20000


Confusion Matrix:
 [[    0  1963     0]
 [    0 17039     0]
 [    0   998     0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## 8. Feature importance for Task 3

In [9]:
import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Top 10 most important features:")
display(feature_importance.head(10))

Top 10 most important features:


,Feature,Importance
9,driver_to_restaurant_distance,0.087414
8,Delivery_Distance_km,0.087085
6,Driver_Lat,0.087020
1,Total_Price,0.086956
2,Restaurant_Lat,0.086800
4,Customer_Lat,0.086023
5,Customer_Lon,0.085961
3,Restaurant_Lon,0.085844
7,Driver_Lon,0.085728
10,order_hour,0.050652


## 9. Task 4 - Feature selection with SelectFromModel

In [10]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(model, prefit=True)

X_train_selected = selector.transform(X_train)
X_test_selected = selector.transform(X_test)

model_selected = RandomForestClassifier(n_estimators=100, random_state=42)
model_selected.fit(X_train_selected, y_train)

y_pred_selected = model_selected.predict(X_test_selected)

print("Accuracy after feature selection:", accuracy_score(y_test, y_pred_selected))
print("\nClassification Report after feature selection:\n", classification_report(y_test, y_pred_selected))
print("\nConfusion Matrix after feature selection:\n", confusion_matrix(y_test, y_pred_selected))

print("\nOriginal number of features:", X_train.shape[1])
print("Selected number of features:", X_train_selected.shape[1])

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Accuracy after feature selection: 0.85195

Classification Report after feature selection:
               precision    recall  f1-score   support

   Cancelled       0.00      0.00      0.00      1963
   Delivered       0.85      1.00      0.92     17039
  In Transit       0.00      0.00      0.00       998

    accuracy                           0.85     20000
   macro avg       0.28      0.33      0.31     20000
weighted avg       0.73      0.85      0.78     20000


Confusion Matrix after feature selection:
 [[    0  1963     0]
 [    0 17039     0]
 [    0   998     0]]

Original number of features: 33
Selected number of features: 10


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
